### Cluster Separability Function

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

path = Path('/content/drive/MyDrive/hydromind')
fig_path = path / "figures/processed"
data_path = path / "data/processed"
fig_path.mkdir(parents=True, exist_ok=True)

def cluster_suitability(feature_matrix: np.ndarray, name: str, max_k: int = 8) -> dict[str, list]:
    # PCA / t-SNE visualization + Elbow + Silhoutte Analysis
    pca = PCA(n_components=min(15, feature_matrix.shape[1]))
    pca_result = pca.fit_transform(feature_matrix)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # scree plot
    axes[0].bar(
        range(1, len(pca.explained_variance_ratio_) + 1),
        pca.explained_variance_ratio_,
        color="teal",
        alpha=0.7
    )
    axes[0].set_title("PCA Scree Plot")
    axes[0].set_xlabel("Principal Component")
    axes[0].set_ylabel("Explained Variance Ratio")

    # t-SNE 2D projection
    tsne_sample_size = min(5000, feature_matrix.shape[0])
    random_indices = np.random.choice(feature_matrix.shape[0], tsne_sample_size, replace=False)
    X_tsne_sample = feature_matrix[random_indices]

    tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
    tsne_result = tsne.fit_transform(X_tsne_sample)

    axes[1].scatter(tsne_result[:, 0], tsne_result[:, 1], s=5, alpha=0.6, c="steelblue")
    axes[1].set_title(f"t-SNE 2D Projection (n={tsne_sample_size})")
    axes[1].axis('off')

    # elbow + silhouette
    inertias, silhouettes = [], []
    for k in range(2, max_k + 1):
        km = KMeans(n_clusters=k, random_state=42, n_init="auto")
        labels = km.fit_predict(feature_matrix)
        inertias.append(km.inertia_)

        score = silhouette_score(feature_matrix, labels, sample_size=2000, random_state=42)
        silhouettes.append(score)

    ax2 = axes[2]
    ax2.plot(range(2, max_k + 1), inertias, "o-", color="crimson", label="Inertia")
    ax2.set_ylabel("Inertia", color="crimson")

    ax2b = ax2.twinx()
    ax2b.plot(range(2, max_k + 1), silhouettes, "s-", color="teal", label="Silhouette")
    ax2b.set_ylabel("Silhouette Score", color="teal")

    ax2.set_xlabel("Number of Clusters (k)")
    ax2.set_title("Elbow + Silhouette Analysis")

    # combine legends
    lines_1, labels_1 = axes[1].get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax2.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper right')

    plt.tight_layout()
    plt.savefig(f"{fig_path}/Cluster Analysis ({name}).png", dpi=150)
    plt.close()

    return {
        "name": name,
        "pca_variance": pca.explained_variance_ratio_.tolist(), 
        "silhouettes": silhouettes
    }

north_final = pl.read_parquet(data_path / "north_final.parquet")
south_final = pl.read_parquet(data_path / "south_final.parquet")

print(cluster_suitability(north_final.to_numpy(), name="North Hemisphere", max_k=8))
print(cluster_suitability(south_final.to_numpy(), name="South Hemisphere", max_k=8))

### Conditional Distribution Analysis

In [ ]:
import polars as pl
import seaborn as sns
from typing import Literal

def conditional_analysis(hemisphere: Literal["north", "soutt"]) -> None:
    # load the newly trained dataset 
    df = pl.read_parquet(data_path / f"{hemisphere}_unscaled.parquet")

    # convert to Pandas exclusively for Seaborn plotting compatibility
    plot_df = df.to_pandas()

    labeled_df = pl.read_parquet(data_path / f"{hemisphere}_final_with_clusters.parquet")
    cluster_ids = labeled_df["cluster_id"]

    plot_df["cluster_id"] = cluster_ids

    # define the 4 independent features
    features = [
        "log_per_capita_usage",
        "dry_day_spike_factor",
        "efficiency_penalty_ratio",
        "landscape_demand_index"
    ]

    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()

    # generate the boxplots dynamically
    for i, feature in enumerate(features):
        sns.boxplot(
            data=plot_df,
            x="cluster_id",
            y=feature,
            palette="viridis",
            ax=axes[i],
            order=["0", "1", "2", "3"] # keeps the X-axis strictly ordered 0-3
        )
    
        clean_title = feature.replace("_", " ").title()
        axes[i].set_title(f"Distribution of {clean_title}", fontsize=14, pad=10)
        axes[i].set_xlabel("Behavioral Archetype (Cluster ID)", fontsize=12)
        axes[i].set_ylabel("Metric Value (Unscaled)", fontsize=12)

    plt.suptitle(f"{hemisphere.capitalize()} Hemisphere: Behavioral Archetype Profiles", fontsize=18, y=1.02, fontweight='bold')
    plt.tight_layout()

    # save the high-res figure for documentation
    plt.savefig(fig_path / f"{hemisphere.capitalize()} Archetype Profiles.png", dpi=150, bbox_inches='tight')
    plt.show()

conditional_analysis(hemisphere="north")
conditional_analysis(hemisphere="south")